
# Polar dust E(B-V) reddens Type 1 & 2 AGN differently

Polar dust *disc attenuation* applies only to Type 1 (face-on) sightlines —
the equatorial torus already screens the disc for Type 2. The bi-conical
polar dust *absorbs* disc photons regardless of viewing angle, however, and
*re-emits* them isotropically as a FIR graybody (Casey 2012). So both Type 1
and Type 2 sweeps show the FIR re-emission bump growing with E(B-V); only
the UV/optical attenuation is gated by sightline.

This reproduces the X-CIGALE polar dust figure (Stalevski et al. 2016;
Yang et al. 2020, Section 2.2.2).

## References
.. [1] M. Stalevski et al., "3D radiative transfer modeling of the dusty
   torus around AGN," MNRAS, 420, 2756 (2012).
   arXiv:1109.1286. https://doi.org/10.1111/j.1365–2966.2011.19775.x
.. [2] W. Yang et al., "X-CIGALE: fitting AGN/galaxy SEDs from X-ray to radio,"
   MNRAS, 491, 740 (2020). arXiv:1909.09632.
   https://doi.org/10.1093/mnras/stz3001


In [ ]:
import warnings

import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

# Physical constants for unit conversion
C_AA_PER_S = 2.998e18

# Load minimal SSP (constant star formation; AGN dominates).
ssp = tengri.load_ssp()

# Suppress stellar/nebular component so the AGN SED is unambiguous.
COMMON = dict(
    sfh={"type": "const", "all_params": tengri.FIXED, "log_total_mass": -30.0},
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0},
    redshift=tengri.Fixed(0.05),
)

# AGN: multicolor disc + SKIRTOR torus + polar-dust attenuation.
# Make agn_polar_ebv and agn_cos_inc FREE so we can sweep them at predict time.
AGN = {
    "disc": {"type": "multicolor", "all_params": tengri.FIXED},
    "torus": {"type": "skirtor", "all_params": tengri.FIXED, "tau_skirtor": 7.0},
    "nlr": {"type": "none", "all_params": tengri.FIXED},
    "blr": {"type": "none", "all_params": tengri.FIXED},
    "atten": {"type": "polar_dust", "all_params": tengri.FIXED},
    "all_params": tengri.FIXED,
    "log_lbol": 12.0,
    "lum_ratio": 1.0,  # without this the AGN is multiplied by 0 (default)
    # A Distribution at per-param level overrides the wildcard and makes the
    # parameter FREE (a bare FREE sentinel here is swallowed by '*: FIXED').
    "polar_ebv": tengri.Uniform(0.0, 0.5),
    "cos_inc": tengri.Uniform(0.0, 1.0),
}

print("Building shared AGN model (polar_ebv and cos_inc FREE)...")
model = tengri.SEDModel.build(ssp, agn=AGN, **COMMON)
params_base = dict(model.spec.sample(jax.random.PRNGKey(42)))

# E(B-V) sweep values
ebv_values = np.array([0.0, 0.05, 0.10, 0.15, 0.20, 0.30])
norm = mpl.colors.Normalize(vmin=ebv_values.min(), vmax=ebv_values.max())
cmap = plt.get_cmap("viridis")

# Create side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.0, 4.5), sharey=True, gridspec_kw={"hspace": 0.0})

# Type 1 sweep (cos_inc = 1.0, polar dust attenuates observed disc)
for ebv in ebv_values:
    params_t1 = {
        **params_base,
        "agn_polar_ebv": jnp.float64(ebv),
        "agn_cos_inc": jnp.float64(1.0),
    }
    out_t1 = model.predict(params_t1)
    wave_t1 = np.asarray(model.wavelengths)
    sed_t1 = np.asarray(out_t1.rest_sed())
    nu_t1 = C_AA_PER_S / wave_t1
    nu_lnu_t1 = nu_t1 * sed_t1
    ax1.loglog(wave_t1, nu_lnu_t1, color=cmap(norm(ebv)), lw=1.4)

ax1.set_xlim(100, 1e6)
ax1.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]", fontsize=11)
ax1.set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]", fontsize=11)
ax1.set_title("Type 1 (face-on, cos θ = 1.0)", fontsize=12, fontweight="bold")
ax1.grid(True, alpha=0.2, which="both")

# Type 2 sweep (cos_inc = 0.0). Disc-attenuation gated off, but polar-dust
# re-emission is bi-conical and isotropic → still grows with E(B-V).
for ebv in ebv_values:
    params_t2 = {
        **params_base,
        "agn_polar_ebv": jnp.float64(ebv),
        "agn_cos_inc": jnp.float64(0.0),
    }
    out_t2 = model.predict(params_t2)
    wave_t2 = np.asarray(model.wavelengths)
    sed_t2 = np.asarray(out_t2.rest_sed())
    nu_t2 = C_AA_PER_S / wave_t2
    nu_lnu_t2 = nu_t2 * sed_t2
    ax2.loglog(wave_t2, nu_lnu_t2, color=cmap(norm(ebv)), lw=1.4)

ax2.set_xlim(100, 1e6)
ax2.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]", fontsize=11)
ax2.set_title("Type 2 (edge-on, cos θ = 0.0)", fontsize=12, fontweight="bold")
ax2.grid(True, alpha=0.2, which="both")

# Colorbar
cbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=[ax1, ax2], pad=0.01, aspect=20
)
cbar.set_label(r"$E(B-V)$ [mag]", fontsize=11)

fig.tight_layout()
plt.savefig("plot_polar_dust_ebv_type12_sweep.png", dpi=150, bbox_inches="tight")
print("Saved: plot_polar_dust_ebv_type12_sweep.png")